# Big Data - Lab 5
**Student:** Manzi Fred  
**ID:** 26634  
**Dataset:** week10_students.xlsx (150 AUCA students)

In [ ]:
#1.1 - Load data and check correlations
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel("week10_students.xlsx")
print(df.shape)
print("correlation of score with quiz_avg : ", df["score"].corr(df["quiz_avg"]))
print("correlation of score with assignments_avg : ", df["score"].corr(df["assignments_avg"]))
print("correlation of score with attendance_pct : ", df["score"].corr(df["attendance_pct"]))

# Most useful feature is quiz_avg (highest correlation ~0.80)

In [ ]:
#1.2 - Prepare features and split into train/test sets
x = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
y = df[["score"]]

#1.2.b - 75% train, 25% test, random_state=42 for reproducibility
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
print("Train : ", len(x_train))
print("Test  : ", len(x_test))

In [ ]:
#2.1 - Train a Linear Regression model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x_train, y_train)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
# quiz_avg has the largest coefficient (~0.58), confirming it drives the score the most

In [ ]:
#2.2 - Evaluate the model on the test set
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(x_test)
print("MAE :", mean_absolute_error(y_test, pred))   # average points we are off by
print("R²  :", r2_score(y_test, pred))              # percentage of score variance explained

In [ ]:
#2.3 - Predict score for a new hypothetical student
new_student = pd.DataFrame([[70, 75, 90]], columns=["quiz_avg", "assignments_avg", "attendance_pct"])
print("New student predicted score : ", model.predict(new_student))

# My own hypothetical profile (strong student)
my_own = pd.DataFrame([[80, 85, 95]], columns=["quiz_avg", "assignments_avg", "attendance_pct"])
print("My own predicted score      : ", model.predict(my_own))

In [ ]:
#3.1.a - Create binary 'passed' label (score >= 50 → 1, else → 0)
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

In [ ]:
#3.1.b - Train a Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

xc = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
yc = df["passed"]

xc_train, xc_test, yc_train, yc_test = train_test_split(xc, yc, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(xc_train, yc_train)

In [ ]:
#3.2.a - Accuracy of the classifier
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred_class = clf.predict(xc_test)
print("Model accuracy : ", accuracy_score(yc_test, y_pred_class))

In [ ]:
#3.2.b - Lazy baseline: always predict 'passed'
print("Lazy baseline (always predict pass) : ", yc_test.mean())
# If this equals model accuracy, the model adds no value beyond guessing

In [ ]:
#3.2.c - Confusion matrix
print("Confusion matrix : ")
print(confusion_matrix(yc_test, y_pred_class))
# Rows = actual (0=fail, 1=pass), Columns = predicted

In [ ]:
#3.2.d - Interpretation
# The model is NOT useless even if accuracy looks similar to baseline.
# The class is 87% passers, so a naive model could also get ~89% accuracy.
# But the confusion matrix shows the model correctly identifies some failing students,
# which is exactly what we need for early-warning intervention.

In [ ]:
#4 - Feature ablation: compare R² for different feature sets
def evaluate_r2(features):
    xt, xte, yt, yte = train_test_split(features, y, test_size=0.25, random_state=42)
    m = LinearRegression()
    m.fit(xt, yt)
    return r2_score(yte, m.predict(xte))

r_quiz_only    = evaluate_r2(df[["quiz_avg"]])
r_all_three    = evaluate_r2(df[["quiz_avg", "assignments_avg", "attendance_pct"]])
x_dummy        = pd.get_dummies(df[["quiz_avg", "assignments_avg", "attendance_pct", "program"]], columns=["program"])
r_with_program = evaluate_r2(x_dummy)

print(f"quiz only          : {r_quiz_only:.4f}")
print(f"all three          : {r_all_three:.4f}")
print(f"all three + program: {r_with_program:.4f}")
# More features are not always better — adding 'program' may not improve R²

In [ ]:
#5.1 - Three Plotly visualizations for the class dashboard
import plotly.express as px

# Chart 1: Score distribution histogram
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

# Chart 2: Who is at risk — quiz_avg vs score, coloured by pass/fail
fig2 = px.scatter(df, x="quiz_avg", y="score", color="passed",
                  hover_name="name", title="2. Who is at risk")
fig2.show()

# Chart 3: Average score by district — where to focus support
avg_by_district = df.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(avg_by_district, x="district", y="score", title="3. Where to act")
fig3.show()

In [ ]:
#5.2 - Export the scatter (at-risk) chart as a standalone HTML dashboard
fig2.write_html("class_dashboard.html")
print("Dashboard saved to class_dashboard.html")

In [ ]:
#Bonus - Score distribution by academic program (box plot)
fig_bonus = px.box(df, x="program", y="score", color="program",
                   title="Score distribution by program")
fig_bonus.show()